In this code I attempt to perform the Olley-Pakes procedure to estimation production function 

In [41]:
# Import packages 
using Pkg 
using GLM
using CSV
using DataFrames
using LinearAlgebra
using Statistics
using DataFrames
using Random
using Distributions
using Optim

In [42]:
# import and browse dataset
dataset = CSV.read("op_ready.csv", DataFrame)

# first(dataset, 5)
data_investment_positive = filter(row -> row.v_investment > 0, dataset)

data_1990 = filter(row -> row.year == 1990, data_investment_positive)

# this data of the year 1990 will be one we use for our first estimation
first(data_1990, 5)


Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor
,Int64,Int64,Float64,Float64,Float64,Float64,Float64
1,168,1990,13.5036,11.0526,11.6892,-0.235295,17.1767
2,169,1990,15.0026,13.0103,12.8541,-0.0655217,19.8712
3,170,1990,15.7607,13.011,13.8029,-0.156504,25.8591
4,171,1990,13.1126,11.7928,12.9358,-0.0875776,12.1316
5,172,1990,16.228,12.9631,12.5244,-0.164247,35.267


Now we proceed with step 1 of Olley-Pakes

I will fit a second-order polynomial on f(capital, material) and perform a regression with production (as dependent variable), labor, capital and investment, and year FE, to find beta_labor and the non-parametric fit.  


In [43]:
# generating the polynomial terms 
data_1990.v_capital_square = data_1990.v_capital .* data_1990.v_material
data_1990.v_material_square = data_1990.v_material .* data_1990.v_material

# display(first(data_1990, 5))

step1_1990 = lm(@formula(v_production ~ v_labor + v_capital + v_material + v_capital_square + v_material_square), data_1990)

display(step1_1990) #print out the labor coefficient

β_labor = coef(step1_1990)[2]

data_1990.production_residuals = data_1990.v_production - β_labor * data_1990.v_labor 

display(first(data_1990, 5))

StatsModels.TableRegressionModel{LinearModel{GLM.LmResp{Vector{Float64}}, GLM.DensePredChol{Float64, CholeskyPivoted{Float64, Matrix{Float64}, Vector{Int64}}}}, Matrix{Float64}}

v_production ~ 1 + v_labor + v_capital + v_material + v_capital_square + v_material_square

Coefficients:
─────────────────────────────────────────────────────────────────────────────────
                        Coef.  Std. Error      t  Pr(>|t|)   Lower 95%  Upper 95%
─────────────────────────────────────────────────────────────────────────────────
(Intercept)         6.17116    0.397597    15.52    <1e-44   5.39005    6.95226
v_labor             0.0214515  0.00220924   9.71    <1e-19   0.0171113  0.0257917
v_capital           0.620267   0.0327859   18.92    <1e-60   0.555857   0.684677
v_material          0.127415   2.48845      0.05    0.9592  -4.7613     5.01613
v_capital_square   -0.0895351  0.183504    -0.49    0.6258  -0.45004    0.27097
v_material_square  -2.05393    2.25933     -0.91    0.3637  -6.492

Row,firm_id,year,v_production,v_investment,v_capital,v_material,v_labor,v_capital_square,v_material_square,production_residuals
,Int64,Int64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,168,1990,13.5036,11.0526,11.6892,-0.235295,17.1767,-2.7504,0.0553639,13.1351
2,169,1990,15.0026,13.0103,12.8541,-0.0655217,19.8712,-0.842221,0.00429309,14.5763
3,170,1990,15.7607,13.011,13.8029,-0.156504,25.8591,-2.1602,0.0244934,15.206
4,171,1990,13.1126,11.7928,12.9358,-0.0875776,12.1316,-1.13288,0.00766983,12.8523
5,172,1990,16.228,12.9631,12.5244,-0.164247,35.267,-2.0571,0.0269769,15.4715


Some comments on step 1 
Descriptions of step 2 next
